The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson8/lab.html

# Lab 8: Find Groups, and Catch It When It Goes Wrong
## MPS311/439 Machine Learning
**Lesson 8 Lab Session | Core: about 43 minutes**

## What this lab is for

The lecture explained why K-means and hierarchical clustering work. Today you use them as tools, on four jobs: find groups in a table of wines, sort the handwritten digits as Assignment 2 asks, cope when there are too many images for hierarchical clustering, and check code that an AI wrote. All the code is given. You run it, change the value marked `← change`, and decide what the numbers tell you. After each Part, **write down** one number from your output and one sentence.

Each Part ends with a folded **Going further** box: open questions with no single answer, for when you finish early. The folded **Try with AI** boxes are also for when you finish early; the **Advanced Study** is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones.
2. Write your answers in a text cell (**+ Text**) under the code.
3. **Run, and you should see** tells you what your output should show, so you know you are on track.

## Part 0: Set up (3 minutes)

Run this cell without reading it closely. It is the same at the start of every lab, with today's new tools at the end.

In [ ]:
import numpy as np                                        # numbers and arrays
import pandas as pd                                       # tables
import matplotlib.pyplot as plt                           # plots
import time                                               # a clock, to time the fitting
import urllib.request                                     # to download a file
from sklearn.datasets import load_wine                    # the first dataset
from sklearn.model_selection import train_test_split, KFold   # splitting the data
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.decomposition import PCA                     # fewer numbers per image
from sklearn.cluster import KMeans, AgglomerativeClustering   # today's tools

## Part 1: Find groups in a table of wines (9 minutes)

**The job.** 178 wines from one region of Italy (the wine data that comes with scikit-learn, a teaching dataset), each with 13 chemical measurements: alcohol, acidity, colour, proline and so on. You want to know whether the wines fall into a few natural groups. The data also say which of three grape varieties each wine comes from; K-means never sees that, and we use it only afterwards, to check. Run the data block as it is: the two `← change` marks only show which lines you would change for another dataset.

In [ ]:
data = load_wine()                                        # ← change: which dataset
X = data.data                                             # the 13 measurements (features)
y = data.target                                           # ← change: the labels (here the grape variety, 0, 1 or 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Run, and you should see:** `(133, 13) [44 53 36]`.

Split the training wines into 3 groups, then count how the groups match the varieties:

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0)      # 3 groups, the best of 10 starting points
groups = km.fit_predict(X_train)                          # ← change: X_train or X_train_s; no labels are passed in
table = np.zeros((3, 3), dtype=int)                       # rows: groups, columns: grape varieties
for i in range(len(groups)):                              # every training wine
    table[groups[i], y_train[i]] += 1                     # add one to its group and its variety
print(table)                                              # the count table
print(round(table.max(axis=1).sum() / len(groups), 3))    # majority vote: each group guesses its most common variety
spread = pd.Series(X_train.std(axis=0), index=data.feature_names)   # the spread of each column
print(spread.sort_values(ascending=False).head(3).round(1))          # the three columns with the largest spread

Run it as it is, then change `X_train` to `X_train_s` (the standardised columns) and run again.

**Write down:** the majority vote of each run. In one sentence: why does the first run do worse? (Look at the three spreads: which column decides the distance between two wines?)

*Your answer:* 

**Think about** (with your neighbour, no writing): the majority vote used the varieties. Without them, how would you know the second grouping is better?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the 3 group centres before and after standardising, as bar charts over the 13 columns.
> 2. **Tell the AI:** `km.cluster_centers_` has 3 rows of 13 numbers; `data.feature_names` holds the column names.
> 3. **Ask:** "Write simple Python, one step per line, that draws the 3 rows of `km.cluster_centers_` as grouped horizontal bars, one group of bars per column name."
> 4. **Check:** for `X_train`, only the proline bars are visible; for `X_train_s`, the centres differ on many columns.

> **Going further (optional)**
>
> - With `X_train_s`, try 1 to 8 groups and print `km.inertia_` for each. Is there a clear bend at 3?
> - Give the 45 test wines to their nearest centre with `km.predict(X_test_s)`, and let each take its group's guess (`table.argmax(axis=1)`). What share is right?


## Part 2: Sort the digits, as Assignment 2 asks (11 minutes)

**The job.** Assignment 2 asks you to run K-means with 10 groups on the digits reduced to 50 components, make a count table and compute the majority vote. Load the digits with the same data block, then split off a validation set (the same three parts as the assignment).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/wayXing/mps311-439-course-materials/main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(DATA_URL, "mnist10k.npz")      # 10,000 handwritten digits, about 1.6 MB (if this fails, see the Appendix)
data = np.load("mnist10k.npz")                            # ← change: which dataset
X = data["X"] / 255                                       # the 784 pixels of each 28 x 28 image, scaled to 0 to 1
y = data["y"]                                             # ← change: the labels (the digit, 0 to 9)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column (not used for the digits today)
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0, stratify=y_train)   # fit on 3/4, check on 1/4
print(X_fit.shape, X_val.shape, X_test.shape)             # fitting, validation, test

**Run, and you should see:** `(7500, 784)` with between 680 and 851 of each digit, then `(5625, 784) (1875, 784) (2500, 784)`.

Learn 50 components from the fitting images only, and split the fitting images into k groups:

In [ ]:
k = 10                                                    # ← change: the number of groups
pca = PCA(n_components=50, random_state=0)                # 50 components, as in Assignment 2
pca.fit(X_fit)                                            # learn the components from the fitting images only
Z_fit = pca.transform(X_fit)                              # 50 numbers for each fitting image
Z_val = pca.transform(X_val)                              # the same directions for the validation images
km = KMeans(n_clusters=k, n_init=10, random_state=0)      # k groups, the best of 10 starting points
km.fit(Z_fit)                                             # no labels are passed in
groups = km.labels_                                       # the group of every fitting image
print(round(km.inertia_, 1))                              # within-cluster sum of squares

**Run, and you should see:** `169025.6`.

Now use the labels, only to check. Each group guesses its most common digit; each validation image goes to its nearest centre and takes that group's guess:

In [ ]:
table = np.zeros((k, 10), dtype=int)                      # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print(table)                                              # the count table
print("guesses:", guess)                                  # one guess per group
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

**Run, and you should see:** a 10 × 10 table, ten guesses, `fitting: 0.576` and `validation: 0.577`.

**Write down:** the two majority votes, and the digits that are no group's guess (look for the digits 0 to 9 missing from `guesses`). In one sentence: why are there digits that no group guesses?

*Your answer:* 

**Think about** (with your neighbour, no writing): the validation images were never used to make the groups. Why do we still check on them?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the centre of each group as a 28 × 28 image.
> 2. **Tell the AI:** `km.cluster_centers_` has k rows of 50 numbers; `pca.inverse_transform` turns them back into 784 pixels; `guess` holds each group's guess.
> 3. **Ask:** "Write simple Python that turns `km.cluster_centers_` back into pixels with `pca.inverse_transform` and shows the rows as 28 × 28 grey images in one row, titled with the group's value of `guess`."
> 4. **Check:** 10 blurred digits; two 0s and two 1s, and the group that guesses 4 looks more like a 9, because it mixes 4s, 7s and 9s.

> **Going further (optional)**
>
> - Change `k = 10` to `k = 15` and run both cells again. Does every digit now get a group? The majority vote goes up: does that mean 15 groups are better? (What is the majority vote with 5625 groups?)
> - Run K-means from single starting points and compare:
> 
>

In [ ]:
for seed in range(5):                                     # five different starting points
    km1 = KMeans(n_clusters=10, n_init=1, random_state=seed)   # only one start
    km1.fit(Z_fit)                                        # run K-means from that start
    t = np.zeros((10, 10), dtype=int)                     # count table for this run
    for i in range(len(Z_fit)):                           # every fitting image
        t[km1.labels_[i], y_fit[i]] += 1                  # add one to its group and its digit
    print(seed, round(km1.inertia_, 1), round(t.max(axis=1).sum() / len(Z_fit), 3))   # seed, within-cluster sum of squares, majority vote

>   Does the run with the smallest sum also have the highest majority vote?


## Part 3: Too many images for hierarchical clustering (7 minutes)

**The job.** Assignment 2 has 50,000 training images. Hierarchical clustering keeps the distance between every pair of images, 8 bytes each. Try it on a random part of the fitting images and see what it costs.

In [ ]:
n = 2000                                                  # ← change: how many fitting images to use
rng = np.random.default_rng(0)                            # fixed random numbers, so everyone picks the same rows
rows = rng.choice(len(Z_fit), n, replace=False)           # n different rows, picked at random
Z_part = Z_fit[rows]                                      # their 50 numbers
y_part = y_fit[rows]                                      # their digits (only to check)
print("memory for every pair:", round(n * (n - 1) / 2 * 8 / 1e9, 3), "GB")   # n(n-1)/2 pairs, 8 bytes each
start = time.time()                                       # the clock before fitting
hc = AgglomerativeClustering(n_clusters=10, linkage="ward")   # 10 groups; Ward: merge where the spread grows least
groups_hc = hc.fit_predict(Z_part)                        # one group per image
print("seconds:", round(time.time() - start, 1))          # how long it took
table_hc = np.zeros((10, 10), dtype=int)                  # rows: groups, columns: true digits
for i in range(n):                                        # every image in the part
    table_hc[groups_hc[i], y_part[i]] += 1                # add one to its group and its digit
print("majority vote:", round(table_hc.max(axis=1).sum() / n, 3))   # each group guesses its most common digit

**Run, and you should see:** `memory for every pair: 0.016 GB`, a time of about a second or less, and `majority vote: 0.62`. Then try `n = 500` and `n = 4000`.

**Write down:** the majority vote for each `n`, and the memory 50,000 images would need (use the formula in the cell; do not run it with 50,000). In one sentence: how would you run hierarchical clustering in Assignment 2?

*Your answer:* 

**Think about** (with your neighbour, no writing): the vote changes with `n` (and would change with the random rows too). How much of a difference between two methods would you trust?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** a dendrogram of 40 random fitting images, with their digits as leaf labels.
> 2. **Tell the AI:** `Z_fit` holds 50 numbers per fitting image; `y_fit` their digits.
> 3. **Ask:** "Write simple Python that picks 40 rows of `Z_fit` with `np.random.default_rng(0)`, computes `scipy.cluster.hierarchy.linkage` with `method='ward'`, and draws the dendrogram with the digits as leaf labels."
> 4. **Check:** 40 leaves; leaves with the same digit often sit next to each other, and the last few merges are much higher than the rest.

> **Going further (optional)**
>
> - Change `linkage="ward"` to `linkage="single"` with `n = 2000`. Print `np.bincount(groups_hc)`: how large is the largest group?


## Part 4: Check the code an AI wrote (12 minutes)

**The job.** You asked an AI for "K-means with 10 groups on the 50 components, the count table and the majority vote on the validation images". It gave three versions. Each differs from Part 2 in one line (named above each version), and each runs without an error. First run this correct version once more, so you have it next to the three. It fits K-means with 10 groups again, in case you changed `k` in Going further:

In [ ]:
km = KMeans(n_clusters=10, n_init=10, random_state=0)     # 10 groups, as in Part 2
km.fit(Z_fit)                                             # no labels are passed in
groups = km.labels_                                       # the group of every fitting image
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

**Run, and you should see:** `fitting: 0.576` and `validation: 0.577`.

**Version A** (the last line is different)

In [ ]:
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((groups_val == y_val).mean(), 3))         # share whose group is their digit

**Version B** (the line that makes `groups_val` is different)

In [ ]:
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(Z_val)   # groups for the validation images
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

**Version C** (the line inside the loop is different)

In [ ]:
table = np.zeros((10, 10), dtype=int)                     # a count table
for i in range(len(groups)):                              # every fitting image
    table[y_fit[i], groups[i]] += 1                       # add one to its digit and its group
guess = table.argmax(axis=1)                              # the most common entry of each row
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

**Write down:** one line per version: its two scores, and a few words on why the changed line is wrong. Very low scores here are real results, not copying mistakes. For each version ask: **what does a group number mean, and which groups does a validation image get?**

*Your answer:* 

**Think about** (with your neighbour, no writing): in Version C, the fitting score alone looked fine. What habit would catch this kind of mistake?

> **Going further (optional)**
>
> - Write your own version D that differs from Part 2 in one line, runs without an error and is wrong (for example, fitting PCA on `X_train`, which contains the validation images). Give it to your neighbour. Can they find the line from the code alone? From the scores alone?


## Part 5: The test set, once (1 minute)

In [ ]:
table = np.zeros((10, 10), dtype=int)                     # the correct count table again (Part 4 may have changed it)
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
groups_test = km.predict(pca.transform(X_test))           # the test images: transform and predict only
print(round((guess[groups_test] == y_test).mean(), 3))    # the 2,500 test images, read once

**Run, and you should see:** `0.577`.

# After class

## How today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 2: K-means with 10 groups on 50 components, fitted on the fitting images only | Section 3, Going further: run K-means (k = 10) on the data reduced to 50 components; which digits does each group contain? (The assignment calls the 50 components "k = 50"; here k is the number of groups.) |
| Part 2: the count table and the majority vote | Extension task B, Table B: the count table of the 10 groups against the 10 digits, and the accuracy when every group predicts its most common digit |
| Part 3: hierarchical clustering on a random part | Extension task B: run hierarchical clustering on only a part of the training set (for example 5,000 images); K-means can use all of it |
| Part 4: the three versions | Check your own Assignment 2 code for these three mistakes |
| Advanced B: groups as features | Extension task B: can the cluster result be used as a feature for a classifier? |

## Reflection

One or two sentences each.

1. Why can we not compare group numbers with digits directly?
2. Why must validation and test images go to the centres learned from the fitting images, instead of being clustered again?
3. When should you standardise before clustering, and when not? Use Part 1 and the digits as your examples.

# Appendix: only if the digits did not download

Use this only if the first cell of Part 2 stopped with a download error. Each cell below gives exactly the same 10,000 images; run one, then go on with the rest of the data block (from `X = data["X"] / 255`).

**Backup 1: a second copy of the course file**, on another server.

In [ ]:
BACKUP_URL = "https://cdn.jsdelivr.net/gh/wayXing/mps311-439-course-materials@main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(BACKUP_URL, "mnist10k.npz")    # the same file, from a different server
data = np.load("mnist10k.npz")                            # the file holds two arrays, X and y
print(data["X"].shape)                                    # images and pixels

**Backup 2: the official MNIST file** (the one Keras uses), from which the course picked its 10,000 images.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
urllib.request.urlretrieve(MNIST_URL, "mnist.npz")        # all 70,000 MNIST images, about 11 MB
full = np.load("mnist.npz")                               # x_train holds the 60,000 training images
rng = np.random.default_rng(0)                            # the random numbers the course used
pick = rng.choice(60000, 10000, replace=False)            # 10,000 of the 60,000 training images
pick = np.sort(pick)                                      # in their original order
images = full["x_train"][pick]                            # the chosen images, 28 x 28 each
X_course = images.reshape(10000, 784)                     # one row of 784 pixels per image
y_course = full["y_train"][pick]                          # their labels
data = {"X": X_course, "y": y_course}                     # the same two arrays as the course file
print(data["X"].shape)                                    # images and pixels

**Run, and you should see:** `(10000, 784)` from either cell.